# Solution — Module 0: Environment & Python Basics

เฉลยแบบฝึกหัดโมดูล 0 — พร้อมคำอธิบายสั้น ๆ ว่าทำไมโค้ดเขียนแบบนี้

## ข้อ 1 — ฟังก์ชันแปลงหน่วย

In [1]:
def celsius_to_kelvin(temp_c):
    """แปลงองศาเซลเซียสเป็นเคลวิน."""
    return temp_c + 273.15


def calibrate(reading, offset, gain):
    """ปรับค่าอ่าน: ลบ offset (ค่าคลาดศูนย์) แล้วคูณ gain (สัมประสิทธิ์สเกล)."""
    return (reading - offset) * gain


assert abs(celsius_to_kelvin(25.0) - 298.15) < 1e-9
assert abs(calibrate(10.05, 0.05, 2.0) - 20.0) < 1e-9
print("OK — ข้อ 1 ผ่าน")

OK — ข้อ 1 ผ่าน


เห็นไหมว่า `calibrate` ตรงกับโมเดลความคลาดเคลื่อนพื้นฐาน: offset = ค่าคลาดศูนย์ (zero error), gain = ค่าคลาดสเกล (span error) — โครงเดียวกับที่จะใช้ปรับสัญญาณเซนเซอร์ตลอดหลักสูตร

## ข้อ 2 — f-string

In [2]:
mean = 24.73186
std = 0.02615

report = f"mean = {mean:.2f} °C, u = {2 * std:.3f} °C (k=2)"
print(report)

assert report == "mean = 24.73 °C, u = 0.052 °C (k=2)"
print("OK — ข้อ 2 ผ่าน")

mean = 24.73 °C, u = 0.052 °C (k=2)
OK — ข้อ 2 ผ่าน


จุดที่ควรสังเกต: `{2 * std:.3f}` — ใส่ expression ใน f-string ได้เลย และ `k=2` คือ coverage factor ที่รายงานความไม่แน่นอนมาตรฐานใช้เรียก 95% coverage

## ข้อ 3 — NumPy

In [3]:
import numpy as np

rng = np.random.default_rng(42)
readings = rng.normal(100.0, 0.8, 1000)
mean = readings.mean()
std = readings.std()
n_outliers = int((np.abs(readings - mean) > 2 * std).sum())

print(f"mean = {mean:.4f}, std = {std:.4f}, outliers(>2σ) = {n_outliers}")

assert readings.shape == (1000,)
assert abs(mean - 100.0) < 0.2
assert abs(std - 0.8) < 0.1
assert 0 <= n_outliers <= 60
print("OK — ข้อ 3 ผ่าน")

mean = 99.9769, std = 0.7910, outliers(>2σ) = 45
OK — ข้อ 3 ผ่าน


สังเกต: รันซ้ำได้ผลเหมือนเดิมเสมอเพราะ `seed=42` — หลักการ "deterministic synthetic data" ของหลักสูตรนี้ และกฎ 2σ ประมาณ ~4.6% ของข้อมูลปกติ ตัวเลขจริงจะอยู่ราว 40–55 ค่า

## ข้อ 4 — dict comprehension

In [4]:
offsets_mv = {"ch1": 0.02, "ch2": -0.05, "ch3": 0.11}

vals = list(offsets_mv.values())
v_mean = sum(vals) / len(vals)
v_std = (sum((v - v_mean) ** 2 for v in vals) / len(vals)) ** 0.5
z_scores = {ch: (v - v_mean) / v_std for ch, v in offsets_mv.items()}

import math

assert set(z_scores) == {"ch1", "ch2", "ch3"}
assert abs(sum(z_scores.values())) < 1e-9
assert all(math.isfinite(v) for v in z_scores.values())
print("OK — ข้อ 4 ผ่าน:", {k: round(v, 2) for k, v in z_scores.items()})

OK — ข้อ 4 ผ่าน: {'ch1': -0.1, 'ch2': -1.17, 'ch3': 1.27}


## ข้อ 5 — แก้ bug

In [5]:
weights = [0.5, 0.3, 0.2]
values = [10.0, 20.0, 30.0]

weighted_mean = sum(w * v for w, v in zip(weights, values)) / sum(weights)

assert abs(weighted_mean - 17.0) < 1e-9
print("OK — ข้อ 5 ผ่าน")

OK — ข้อ 5 ผ่าน


Bug สองจุดในข้อ 5: (1) คูณค่าด้วยน้ำหนักก่อนรวม — `sum(values)/len(values)` คือค่าเฉลี่ยธรรมดา ไม่ใช่ weighted; (2) หารด้วย `sum(weights)` ไม่ใช่ `len(values)` — ถ้าน้ำหนักรวมไม่เท่า 1 ผลจะคลาดทันที